# 🏃 Runnables — LangChain's Core Interface

## What is a Runnable?
Every component in LangChain implements the **Runnable** interface.

This means every component has the SAME methods:
- `.invoke(input)` — single call
- `.stream(input)` — streaming
- `.batch(inputs)` — parallel processing
- `.ainvoke(input)` — async invoke
- `.astream(input)` — async streaming
- `.abatch(inputs)` — async batching

## Why This Matters
Since EVERYTHING is a Runnable, you can:
- Swap components freely: `llm_a | parser` → `llm_b | parser`
- Build generic utilities that work with any component
- Test components in isolation

## Runnable Hierarchy
```
Runnable (abstract base)
├── ChatOpenAI          (LLM)
├── PromptTemplate      (Prompt)
├── StrOutputParser     (Parser)
├── RunnableSequence    (Chain via |)
├── RunnableParallel    (Parallel execution)
├── RunnableLambda      (Custom Python function)
├── RunnableBranch      (Conditional routing)
└── RunnablePassthrough (Identity)
```


In [ ]:
# ── The Runnable Interface ───────────────────────────────────────────────
from langchain_core.runnables import Runnable
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model='gpt-4o-mini')
prompt = ChatPromptTemplate.from_template('{input}')
parser = StrOutputParser()

# ALL of these are Runnables
components = [llm, prompt, parser]
for comp in components:
    is_runnable = isinstance(comp, Runnable)
    print(f'{type(comp).__name__:30} is Runnable: {is_runnable}')

# They all share the same interface
print('\nShared methods on ChatOpenAI:')
runnable_methods = ['invoke', 'stream', 'batch', 'ainvoke', 'astream', 'abatch']
for method in runnable_methods:
    has_it = hasattr(llm, method)
    print(f'  .{method}(): {has_it}')

In [ ]:
# ── RunnableConfig: Control Execution ──────────────────────────────────
# WHY: RunnableConfig lets you control HOW a chain runs.
# Useful for: timeouts, tags, metadata, callbacks, max concurrency.

from langchain_core.runnables import RunnableConfig
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

chain = (
    ChatPromptTemplate.from_template('What is {topic}?')
    | ChatOpenAI(model='gpt-4o-mini')
    | StrOutputParser()
)

# Pass config to control execution
config = RunnableConfig(
    tags=['production', 'v2'],       # for filtering in LangSmith
    metadata={'user_id': 'user_123'}, # custom metadata
    run_name='topic_explainer',       # name in traces
)

result = chain.invoke({'topic': 'transformers'}, config=config)
print('Result:', result[:100], '...')

In [ ]:
# ── RunnableWithFallbacks: Resilience Pattern ───────────────────────────
# WHY: In production, APIs fail. Fallbacks provide resilience.
# If primary LLM fails, automatically try backup LLM.

from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic  # pip install langchain-anthropic
from langchain_core.output_parsers import StrOutputParser

# Primary LLM
primary = ChatOpenAI(model='gpt-4o')

# Backup LLM (if primary fails)
backup = ChatOpenAI(model='gpt-4o-mini')

# Chain with fallback
resilient_llm = primary.with_fallbacks([backup])

print('Resilient LLM type:', type(resilient_llm).__name__)
print('This will try gpt-4o first, then gpt-4o-mini if it fails.')
print('Use this pattern in production for critical paths.')

In [ ]:
# ── Retry Logic ─────────────────────────────────────────────────────────
# WHY: API calls can fail due to rate limits or network issues.
# with_retry() automatically retries on failure.

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-4o-mini')

# Add retry logic
llm_with_retry = llm.with_retry(
    stop_after_attempt=3,          # max 3 attempts
    wait_exponential_jitter=True,  # exponential backoff + jitter
)

print('LLM with retry configured.')
print('Attempt 1 fails → wait 1s → Attempt 2 fails → wait 2s → Attempt 3')
print('Jitter adds randomness to prevent thundering herd.')

## 🎯 Interview Questions

1. **What is the Runnable interface in LangChain?**
2. **Name 5 Runnable subclasses and their purposes.**
3. **What is RunnableConfig used for?**
4. **How do you add fallback LLMs in LangChain?**
5. **What is exponential backoff with jitter?**

> Answer these before moving to the next module.

## 💪 Mini Exercise

**Exercise**: Based on what you learned in this module:

1. Re-run all code cells with your own inputs
2. Modify one code cell to add new functionality
3. Answer the interview questions above from memory

**Mini Project**: Build a small application that combines the concepts from this module.


## 📚 Summary

In this module you learned:
- The core concepts of **Runnables — LangChain's Core Interface**
- How to implement them with modern LangChain/LangGraph APIs
- Common mistakes and how to avoid them
- Interview-level understanding of why each component exists

**Next**: Continue to the next module to build on these foundations.

---
*Part of the LangChain & LangGraph Master Course*
